# Reference copy - needs non-public data

이 노트북은 원본 연구 노트북의 **참고용 사본**입니다. 실행에 필요한 월별 초과수익률(`excess_return.csv`)과 팩터(`factors_monthly.csv`) 파일은(는) 유료/비공개 데이터라 이 저장소에 포함하지 않았으므로 **그대로는 끝까지 실행되지 않습니다.** 출력, 실행 카운트, 로컬 경로는 제거했습니다. 실행 가능한 데모는 `00_public_sample_workflow.ipynb`(합성 데이터)를 사용하세요.

원본은 학회 세션 자료(단계별 폴더)였고, 파일 경로는 `./input/`, `./output/` 기준의 중립 경로로 바꿨습니다. 이 노트북의 출력 `./output/ivol.csv`는 02 노트북에서 `./input/ivol.csv`로 읽습니다(파일 복사 필요).

---
### **Factor construction (ivol 전략)**

**Idiosyncratic Volatility (ivol) 계산 방법**

- **모형**: Carhart 4-factor  
- **데이터**: 각 종목의 월별 초과수익률  
- **방법**: 60개월 이동 윈도우(rolling window)로 회귀 분석  
- **회귀식**:

$$
R_{i,t} - R_{f,t} 
= \alpha_i 
+ \beta_{MKT}(MKT_t) 
+ \beta_{SMB}(SMB_t) 
+ \beta_{HML}(HML_t) 
+ \beta_{MOM}(MOM_t) 
+ \epsilon_{i,t}
$$

- $ \epsilon_{i,t} $ : 잔차, 개별 종목의 고유 수익률 (idiosyncratic return)


- **ivol 정의**: 각 윈도우별 잔차 $ \epsilon_{i,t} $ 의 표준편차  

$$
ivol_{i,t} = \operatorname{std}\left(\epsilon_{i, t-59:t}\right)
$$

- **전략 적용**: 저 ivol(저변동성) 종목에 롱 포지션, 고 ivol(고변동성) 종목에 숏 포지션

---
**데이터 로드**

In [ ]:
import pandas as pd

# Investment Factor 계산에 필요한 데이터를 input 폴더에서 불러옴
excess_return    = pd.read_csv("./input/excess_return.csv", index_col=0, parse_dates=True)     # 초과수익률 (종속변수)
factors          = pd.read_csv("./input/factors_monthly.csv", index_col=0, parse_dates=True)   # 팩터(MKT, SMB, HML, MOM, RF) (독립변수)

---
**독립변수 설정**

In [ ]:
# 수익률 분해를 위한 팩터 정리 - 독립변수 설정
factors['MKT'] = factors['KOSPI'] - factors['RF']            # MKT 팩터(시장 포트폴리오 초과 수익률)
X = factors[['MKT', 'SMB', 'HML', 'MOM']].copy()

In [ ]:
X.tail()

---
**종속변수 설정**

In [ ]:
excess_return.tail()

---
**ivol 계산 함수 정의**

- **방법**: 60개월 이동 윈도우(rolling window)로 회귀 분석  
- **회귀식**:

$$
R_{i,t} - R_{f,t} 
= \alpha_i 
+ \beta_{MKT}(MKT_t) 
+ \beta_{SMB}(SMB_t) 
+ \beta_{HML}(HML_t) 
+ \beta_{MOM}(MOM_t) 
+ \epsilon_{i,t}
$$

- $ \epsilon_{i,t} $ : 잔차, 개별 종목의 고유 수익률 (idiosyncratic return)


- **ivol 정의**: 각 윈도우별 잔차 $ \epsilon_{i,t} $ 의 표준편차

$$
ivol_{i,t} = \operatorname{std}\left(\epsilon_{i, t-59:t}\right)
$$

In [ ]:
import statsmodels.api as sm
import numpy as np

# iVOL 계산 함수 정의
def rolling_ivol(y: pd.Series,
                 X: pd.DataFrame,
                 window: int = 60) -> pd.Series:
    """
    개별 종목의 idiosyncratic volatility (iVOL) 시계열을 계산.
    
    Parameters
    ----------
    y : pd.Series
        종목의 초과수익률 (index = datetime).
    X : pd.DataFrame
        팩터 수익률 (MKT, SMB, HML, MOM).
    window : int, default 60
        롤링 회귀 구간(개월 수).
    
    Returns
    -------
    pd.Series
        각 시점별 iVOL (60개월 잔차의 표준편차).
    """
    df = pd.concat([y.rename("y"), X], axis=1).dropna()
    ivol = pd.Series(index=y.index, dtype=float)

    for t in range(len(df)):
        end_idx = df.index[t]
        start_pos = max(0, t - window + 1)
        train = df.iloc[start_pos:t+1]

        if len(train) < window:
            continue

        # 회귀
        y_win = train['y']
        X_win = sm.add_constant(train[['MKT','SMB','HML','MOM']], has_constant='add')
        model = sm.OLS(y_win.values, X_win.values).fit()

        # 해당 윈도우 잔차의 표준편차 = iVOL
        resid_std = np.std(model.resid, ddof=1)
        ivol.loc[end_idx] = resid_std

    return ivol

---
**ivol 계산(병렬 처리를 이용)**

In [ ]:
from joblib import Parallel, delayed
from tqdm import tqdm

tickers = excess_return.columns.tolist()

# 병렬로 iVOL 계산
ivol_list = Parallel(n_jobs=-1)(
    delayed(rolling_ivol)(excess_return[ticker], X, window=60)
    for ticker in tqdm(tickers, desc="Calculating iVOL")
)

ivol_df = pd.concat(ivol_list, axis=1)
ivol_df.columns = tickers

In [ ]:
ivol_df.tail()

---
**데이터 저장**

In [ ]:
# Output 폴더
ivol_df.to_csv("./output/ivol.csv", index=True, encoding="utf-8-sig")

# 다음 단계(02)에서 사용
# 02 노트북이 읽는 위치: ./input/ivol.csv (원본에서는 다음 단계 폴더로 복사)